In [0]:
from pyspark.sql.functions import col
from pyspark.sql.functions import lit

spark.conf.set(
  "fs.azure.account.key.adlsvenproject1.dfs.core.windows.net",
  dbutils.secrets.get(scope="secret-scope-azure-live-project-1", key="storage-key")
)

silver_path = "abfss://datalake-ven-project1@adlsvenproject1.dfs.core.windows.net/silver/sales_data/"
df_silver = spark.read.format("delta").load(silver_path)

In [0]:
from pyspark.sql.functions import sum, countDistinct

df_gold = df_silver.groupBy("OrderDate").agg(
    countDistinct("SalesOrderID").alias("TotalOrders"),
    sum("OrderQty").alias("TotalQuantity"),
    sum("LineTotal").alias("TotalRevenue"),
    countDistinct("CustomerID").alias("UniqueCustomers")
)

df_gold = df_gold.filter("OrderDate IS NOT NULL")

In [0]:
gold_path = "abfss://datalake-ven-project1@adlsvenproject1.dfs.core.windows.net/gold/sales_summary/"

df_gold.write.format("delta") \
    .mode("overwrite") \
    .save(gold_path)

In [0]:
df_gold.display()

In [0]:
from pyspark.sql.functions import sum

# Silver total
silver_total = spark.read.format("delta").load(silver_path) \
    .agg(sum("LineTotal").alias("total")).collect()[0]["total"]

# Gold total
gold_total = spark.read.format("delta").load(gold_path) \
    .agg(sum("TotalRevenue").alias("total")).collect()[0]["total"]

print("Silver total:", silver_total)
print("Gold total:", gold_total)

In [0]:
df_gold = spark.read.format("delta").load(gold_path)
df_gold.count()

In [0]:
spark.read.format("delta").load(silver_path) \
    .select("OrderDate").distinct().count()

In [0]:
df_gold.filter("OrderDate = '2011-06-12'").display()

In [0]:
spark.read.format("delta").load(silver_path) \
    .filter("OrderDate = '2011-06-12'") \
    .groupBy("OrderDate") \
    .agg(sum("LineTotal")) \
.display()

In [0]:
spark.sql("OPTIMIZE delta.`abfss://datalake-ven-project1@adlsvenproject1.dfs.core.windows.net/gold/sales_summary` ZORDER BY (OrderDate)")